# Pricing Experiments & Contribution Margin

**Pregunta:** ¿un descuento que aumenta ingresos también aumenta el margen?

**SIMULACIÓN independiente:** 800 unidades experimentales, semilla 2427. No corresponde a un experimento en la empresa del dataset UCI. Se conserva el escenario enriquecido de Week 4, con cantidades y costes; no se reutiliza el lift de Week 3.

## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of retail-analytics-pipeline.")
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports"


**Dónde:** Jupyter en Codespaces o una copia del repositorio. La carpeta raíz se identifica por `pyproject.toml`; los mismos módulos se usan desde `main.py`.

## 2. Generar y revisar el escenario

In [2]:
from market_insights.data import simulate_pricing
experiment, costs = simulate_pricing()
display(experiment.head())
display(costs)
display(experiment.groupby('group').size().rename('units'))

,group,revenue_usd,product_category,units_sold,experimental_unit_id
0,treatment,267.72,Dairy,97,EXP-0001
1,treatment,95.40,Beverages,61,EXP-0002
2,control,167.50,Snacks,67,EXP-0003
3,control,392.70,Personal Care,77,EXP-0004
4,treatment,616.03,Frozen Foods,108,EXP-0005


,product_category,unit_cost_usd
0,Frozen Foods,2.790
1,Personal Care,2.295
2,Household Cleaning,1.845
3,Dairy,1.350
4,Snacks,1.125
5,Beverages,0.765


group
control      400
treatment    400
Name: units, dtype: int64

**Stack:** NumPy y pandas. Cada fila representa una unidad experimental independiente. El tratamiento aplica 8% de descuento y un supuesto de +22% de unidades sobre la base generada. El coste unitario equivale a 45% del precio regular. Son supuestos, no impactos reales.

## 3. Comparar revenue, unidades y margen

In [3]:
from market_insights.pricing import evaluate_pricing
results, enriched = evaluate_pricing(experiment, costs)
display(results.round(4))
display(enriched[['revenue_usd','units_sold','unit_cost_usd','margin_usd']].head())

,metric,control_mean,treatment_mean,difference,lift_pct,ci95_low,ci95_high,p_value,p_holm
0,revenue_usd,322.1368,352.5799,30.4432,9.4504,5.9625,54.9239,0.0149,0.0297
1,units_sold,84.9400,101.5875,16.6475,19.5991,12.5617,20.7333,0.0000,0.0000
2,margin_usd,177.1752,180.1224,2.9472,1.6634,-10.0339,15.9283,0.6560,0.6560


,revenue_usd,units_sold,unit_cost_usd,margin_usd
0,267.72,97,1.350,136.770
1,95.40,61,0.765,48.735
2,167.50,67,1.125,92.125
3,392.70,77,2.295,215.985
4,616.03,108,2.790,314.710


**Stack:** pandas y SciPy. El margen de contribución resta unidades × coste de los ingresos; no incluye todos los costes operativos. Welch compara medias y calcula intervalos de 95%. Holm ajusta los tres p-values.

## 4. Visualizar incertidumbre

In [4]:
from market_insights.visualization import plot_pricing
plot_pricing(results, REPORTS/'figures')

**Stack:** Matplotlib. Los puntos representan lifts observados. Los intervalos de diferencias absolutas se expresan como porcentaje de la media de control; no son intervalos estadísticos de cocientes.

![Figura de la ejecución registrada](../reports/figures/pricing_effects.png)

## 5. Recomendación calculada

In [5]:
margin = results.loc[results.metric.eq('margin_usd')].iloc[0]
if margin.difference > 0 and margin.p_holm < 0.05:
    print('La simulación aporta evidencia de aumento del margen; validar un experimento real y costes completos.')
else:
    print('No hay evidencia suficiente de aumento del margen; no justificar el descuento solo por revenue.')

No hay evidencia suficiente de aumento del margen; no justificar el descuento solo por revenue.


La ejecución registrada aumenta revenue y unidades, pero el intervalo de margen incluye cero. Esa distinción es la historia de negocio del análisis; no se afirma un ahorro ni un aumento de beneficio en producción.